<a href="https://colab.research.google.com/github/amit-sw/colab_notebooks/blob/main/20260308_Chatbot.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Set up

In [18]:
!pip install langchain_openai langgraph pydantic langchain_core langchain_community --quiet

In [19]:
from IPython.display import display, Markdown
from langchain_openai import ChatOpenAI
from langgraph.graph import StateGraph, START, END
from pydantic import BaseModel

from langchain_core.messages import SystemMessage, BaseMessage, HumanMessage, AIMessage

In [20]:
import os
from google.colab import userdata

LANGCHAIN_API_KEY=userdata.get('LANGCHAIN_API_KEY')
OPENAI_API_KEY=userdata.get('OPENAI_API_KEY')


os.environ["LANGCHAIN_TRACING_V2"]="true"
os.environ["LANGCHAIN_API_KEY"]=LANGCHAIN_API_KEY
os.environ['OPENAI_API_KEY']=OPENAI_API_KEY
os.environ["LANGCHAIN_PROJECT"]="AIClub_Pro"
os.environ['LANGCHAIN_ENDPOINT']="https://api.smith.langchain.com"

# Single message : Request-response

In [21]:
# Single message
system_prompt="You are a helpful assistant. Please answer the user's questions in 2-3 paragraphs max."
user_input="what is the meaning of life?."

In [22]:
msg_list=[SystemMessage(content=system_prompt),
          HumanMessage(content=user_input)]
model=ChatOpenAI(model_name="gpt-5-nano", openai_api_key=OPENAI_API_KEY)
resp = model.invoke(msg_list)
display(Markdown(resp.content))


There isn’t a single universal answer. Different traditions and perspectives offer different paths: religious or spiritual views often see meaning as something given by a higher power or cosmic plan; existentialist ideas suggest meaning isn’t handed to us but created through our choices and authentic living; secular or humanist approaches find meaning in relationships, personal growth, curiosity, and contributing to others.

If you’re exploring what it means for you, start with your values and what feels truly worthwhile. Reflect on moments when you felt most alive, identify the values those moments reveal, and try small, concrete actions aligned with them—time with loved ones, learning something new, helping others, or creating something meaningful. Meaning often grows from consistent, purposeful action and connection. If you share a bit about your beliefs or what you’re drawn to, I can tailor suggestions.

# Single message: Streaming response

In [23]:
msg_list=[SystemMessage(content=system_prompt),
          HumanMessage(content=user_input)]
model=ChatOpenAI(model_name="gpt-5-nano", openai_api_key=OPENAI_API_KEY)
resp = model.stream(msg_list)

full_text = ""
handle = display(Markdown(""), display_id=True)

for chunk in resp:
    if chunk.content:
        full_text += chunk.content
        handle.update(Markdown(full_text))

There isn’t a single universal answer to the meaning of life. It’s something many people decide for themselves based on their experiences, values, and goals. Some find meaning through relationships, love, and helping others; others through personal growth, curiosity, or creative or professional projects.

Different frameworks offer different directions: religious or spiritual traditions often frame meaning as alignment with a higher purpose or divine plan. Secular or existential perspectives suggest we grant life meaning through the choices we make, the commitments we uphold, and how authentically we live. Some scientists and thinkers argue life may not have intrinsic purpose, but we can still create significance in how we relate to others, pursue understanding, and contribute to the world.

If you’re exploring your own meaning, start with small steps: identify what matters most to you, invest in meaningful relationships, pursue activities that feel authentic, and look for ways to contribute to something larger than yourself. Your sense of meaning can evolve over time, and that evolution is a natural part of being human.

# Chat with memory: Streaming response

In [24]:
msg_list=[SystemMessage(content=system_prompt)]
model=ChatOpenAI(model_name="gpt-5-nano", openai_api_key=OPENAI_API_KEY)
while True:
  user_input=input("User: ")
  if(len(user_input)<=0):
    print("\n\n\nGoodbye!!\n\n\n")
    break
  msg_list.append(HumanMessage(content=user_input))
  resp = model.stream(msg_list)

  full_text = ""
  handle = display(Markdown(""), display_id=True)

  for chunk in resp:
      if chunk.content:
          full_text += chunk.content
          handle.update(Markdown(full_text))
  print("\n**********\n")

User: Who is this?


I can’t identify people in photos. If you share a name or some context (for example, a public figure, character, or event), I can provide information about them.

If you’d like, you can upload the image and I can describe non-identifying details (clothing, setting, objects) or help with related information. What would you like to know or do?


**********

User: 



Goodbye!!





# Graph

In [25]:
def create_llm_msg(system_prompt,history):
    resp=[SystemMessage(content=system_prompt)]
    msgs = history
    for m in msgs:
        if m["role"] == "user":
            resp.append(HumanMessage(content=m["content"]))
        elif m["role"] == "assistant":
            resp.append(AIMessage(content=m["content"]))
    #print(f"DEBUG CREATE LLM MSGS: {history=}\n{resp=}")
    return resp

In [26]:
class AgentState(BaseModel):
    """State of the agent."""
    messages: list = []
    response: str = ""
    category: str = ""

class Category(BaseModel):
    """Category for the agent."""
    category: str

In [31]:
order_db=[
    {'id':1,"status":"Delivered","comments":"Signed by Alice on Jan 1,2026"},
    {'id':2,"status":"In Transit","comments":"Expected: Apr 1, 2026"},
    {'id':3,"status":"Lost","comments":"Reach us at 1-800-123-4567"},
]

In [32]:
def get_order_information(order_id:int):
  for order in order_db:
    if order["id"]==order_id:
      return order
  return None


In [41]:
class ChatbotAgent():
    """A chatbot agent that interacts with users."""

    def __init__(self, api_key: str):
        self.model = ChatOpenAI(model_name="gpt-5-nano", openai_api_key=api_key)
        workflow = StateGraph(AgentState)
        workflow.add_node("classifier", self.classifier)
        workflow.add_node("smalltalk_agent", self.smalltalk_agent)
        workflow.add_node("complaint_agent", self.complaint_agent)
        workflow.add_node("status_agent", self.status_agent)
        workflow.add_node("feedback_agent", self.feedback_agent)
        workflow.add_edge(START, "classifier")
        workflow.add_conditional_edges("classifier", self.main_router)
        #workflow.add_edge("classifier", "smalltalk_agent")
        #workflow.add_edge("classifier", "complaint_agent")
        #workflow.add_edge("classifier", "status_agent")
        #workflow.add_edge("classifier", "feedback_agent")
        workflow.add_edge("smalltalk_agent", END)
        workflow.add_edge("complaint_agent", END)
        workflow.add_edge("status_agent", END)
        workflow.add_edge("feedback_agent", END)

        self.graph = workflow.compile()



    def classifier(self, state: AgentState):
        #print("Initial classsifier")
        messages=state.messages
        CLASSIFIER_PROMPT = """
        You are a helpful assistant that classifies user messages into categories.
        Given the following messages, classify them into one of the following categories:
        - smalltalk_agent
        - complaint_agent
        - status_agent
        - feedback_agent

        If you don't know the category, classify it as "smalltalk_agent".
        """
        llm_messages = create_llm_msg(CLASSIFIER_PROMPT, state.messages)
        llm_response = self.model.with_structured_output(Category).invoke(llm_messages)
        category=llm_response.category
        print(f"Classified category: {category}")
        return {"category":category}

    def main_router(self, state: AgentState):
        #print("Routing to appropriate agent based on category")
        #print(f"DEBUG: Current state: {state}")
        #print(f"DEBUG: Current category: {state.category}")
        return state.category

    def smalltalk_agent(self, state: AgentState):
        print("Smalltalk agent processing....")
        SMALLTALK_PROMPT = f"""
        You are a friendly and engaging smalltalk agent designed to have casual conversations.
        Your purpose is to provide a warm and approachable interaction, ask follow-up questions to keep the conversation flowing,
        and inject a bit of humor or insight when appropriate. Avoid giving overly formal or short responses.
        The goal is to be professional - do not bring up any topics outside of work. Get them to the point without being curt.
        If the user asks about something complex or outside the scope of small talk, gently redirect them to a more specific agent
        (e.g., 'That sounds like a question for our support team, but how's your day going otherwise?').
        Given the following messages, respond appropriately to the user's message, focusing on maintaining a pleasant chat.
        """
        llm_messages = create_llm_msg(SMALLTALK_PROMPT, state.messages)
        return {"response": self.model.stream(llm_messages), "category": "smalltalk_agent"}

    def complaint_agent(self, state: AgentState):
        print("Complaint agent processing....")
        COMPLAINT_PROMPT = f"""
        You are a dedicated complaint resolution agent. Your primary goal is to listen empathetically,
        acknowledge the user's frustration, and gather all necessary details to escalate or resolve their issue.
        Ensure you apologize for any inconvenience caused and reassure the user that their concern is being taken seriously.
        Ask clarifying questions to understand the full scope of the complaint.
        If possible, outline the next steps you will take to address their complaint.
        Given the following messages, respond appropriately to the user's complaint with a focus on resolution and empathy.
        """
        llm_messages = create_llm_msg(COMPLAINT_PROMPT, state.messages)
        return {"response": self.model.stream(llm_messages), "category": "complaint_agent"}

    def status_agent(self, state: AgentState):
        print("Status agent processing....")

        import json
        from pydantic import BaseModel, Field

        class OrderLookup(BaseModel):
            orderid: int = Field(description="The order ID to look up")

        EXTRACT_PROMPT = """
        You extract the order ID from the user's message.
        Return only the order ID.
        """

        extract_messages = create_llm_msg(EXTRACT_PROMPT, state.messages)
        extracted = self.model.with_structured_output(OrderLookup).invoke(extract_messages)
        orderid = extracted.orderid

        order_info = get_order_information(orderid)
        order_info_json = json.dumps(order_info, indent=2)

        STATUS_PROMPT = f"""
        You are a status agent that provides updates on user requests.

        Order ID: {orderid}

        Order lookup result:
        {order_info_json}

        Answer the user clearly.
        State the order status.
        Include any useful commentary.
        If the user has not provided an Order Number, ask them to provide it now.
        """

        llm_messages = create_llm_msg(STATUS_PROMPT, state.messages)
        return {
            "response": self.model.stream(llm_messages),
            "category": "status_agent"
        }

    def feedback_agent(self, state: AgentState):
        print("Feedback agent processing....")
        FEEDBACK_PROMPT = f"""
        You are an attentive feedback collection agent. Your main objective is to solicit and record user feedback
        in a structured and encouraging manner. Thank the user for their input, whether positive or negative,
        and assure them that their feedback is valuable and will be considered for product/service improvement.
        If appropriate, ask follow-up questions to get more detailed insights into their experience.
        Given the following messages, respond appropriately to the user's feedback, showing appreciation for their input.
        """
        llm_messages = create_llm_msg(FEEDBACK_PROMPT, state.messages)
        return {"response": self.model.stream(llm_messages), "category": "feedback_agent"}

# Single Message - Graph

In [42]:
import uuid

In [43]:
# Single message
msg_list=[]

user_input="what is the status of my order?."
msg_list.append({"role":"user","content":user_input})
app=ChatbotAgent(OPENAI_API_KEY)
thread_id = uuid.uuid4()
thread={"configurable":{"thread_id":thread_id}}
full_resp = ""
for s in app.graph.stream({'messages': msg_list}, thread):
  for k,v in s.items():
    if resp_gen := v.get("response"):
      print(f"Assistant: ")
      for chunk in resp_gen:
        text = getattr(chunk, "content", None) or getattr(chunk, "delta", None) # or str(chunk)
        if text:
          print(text, end="", flush=True)
          full_resp += text

if full_resp:
  msg_list.append({"role":"assistant","content":full_resp})

Classified category: status_agent
Status agent processing....
Assistant: 
I don’t have an order number to look up your status. Please provide your order number so I can pull the latest update for you. 

Note: if you recently placed an order, it may take a few minutes to appear in the system.

# Chat - Graph

In [44]:
msg_list=[]
while True:
  user_input=input("User: ")
  if(len(user_input)<=0):
    print("\n\n\nGoodbye!!\n\n\n")
    break
  msg_list.append({"role":"user","content":user_input})
  app=ChatbotAgent(OPENAI_API_KEY)
  thread_id = uuid.uuid4()
  thread={"configurable":{"thread_id":thread_id}}
  full_resp = ""
  for s in app.graph.stream({'messages': msg_list}, thread):
    for k,v in s.items():
      if resp_gen := v.get("response"):
        print(f"Assistant: ")
        for chunk in resp_gen:
          text = getattr(chunk, "content", None) or getattr(chunk, "delta", None) # or str(chunk)
          if text:
            print(text, end="", flush=True)
            full_resp += text

  if full_resp:
    msg_list.append({"role":"assistant","content":full_resp})
    print("\n**********\n")

User: What is the status of my order?
Classified category: status_agent
Status agent processing....
Assistant: 
I can help, but I don’t have an order number to look up yet. Please provide your order number (from the confirmation email or your order history). Once I have it, I’ll tell you the current status and any latest updates.

If you don’t have the number, you can share the email used for the order or your name and the approximate date, and I’ll try to locate it.
**********

User: 2
Classified category: status_agent
Status agent processing....
Assistant: 
Order 2 status: In Transit.

- Expected delivery: April 1, 2026.

No further updates are available right now. Would you like me to set up alerts if the status or delivery date changes?
**********

User: 



Goodbye!!



